# 48 · Georeferenced map image, checked against GDAL

## What this checks

**Georeferenced map image** puts an old map or plan in its place on today's map. One way is to match points: the
same feature, picked on the image and on today's map. This check covers release 0.8.1.

From the matched points, Fieldwork fits an affine placement by least squares. Affine means the image may be moved,
turned, stretched and slanted, but stays flat and straight. Least squares means the placement makes the squared
errors over all the points as small as it can. The placement comes down to six numbers. Each point's error is how far
its match lands from where it should.

A badly placed map puts old cases in the wrong streets. So it is worth checking.

## How it is checked

We give the same points to **GDAL**, a widely used library for geographic data. We use its own fit from
matched points, called ground control points, through rasterio, its Python link. Both sides count pixels the same
way, from the image's top-left corner. No code is shared with Fieldwork.

There are two sets of points:

- **South London.** The four points recorded for Snow's Map 2.
- **A turned set.** Five points on a turned image, with one pair 60 m off on purpose.

## What should happen

1. **The same six numbers**, to 10⁻¹⁰ degree a pixel, and to 10⁻⁷ degree at the image's corner.
2. **The same errors** at each point, worked out here from GDAL's placement, to 1 centimetre.

These criteria were fixed before the check was run.

In [ ]:
NEEDED = ["rasterio"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the installed packages.")

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_48_georeference import run

report = run(Path.cwd() / "fixtures" / "georef-001.json")
print("agrees:", report["agrees"])
for s in report["sets"]:
    worst = max(s["coefficientGap"].values())
    print(f"{s['name']}: {s['points']} points, largest coefficient difference {worst:.1e}, errors (m) {s['residualsM']}")
print("differs:", report["differs"])

## The result on record

First recorded on 10 November 2026, against release 0.3.0. Rerun on 26 November 2026, against release 0.8.1:
**Agrees**.

The six numbers match GDAL's to about 10⁻¹⁷. The errors match to 4 × 10⁻¹⁰ m. In South London the four errors are
129, 79, 35 and 85 m. In the turned set, the bad pair is 47 m off, against 12 m for the rest.

This settles an earlier question. Fieldwork's fit is GDAL's: it predicts coordinates from pixels. South London's
placement made by hand predicted pixels from coordinates. That is why the two differ east to west.

The result agrees in CPython and in the browser kernel.

Running the cell above repeats the check on the same data.

## What this does not show

- That the points are the right features. Both sides fit the same pairs.
- That a plan is accurate. The errors measure agreement with one placement, not how well the surveyor drew the place.
- The north-up placement. GDAL's fit is affine. Fieldwork's north-up fit is checked against the hand placement in Fieldwork's own tests.
- The drawing. This checks the placement, not how the image is drawn on the map.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
e = report["environment"]
print("GDAL:", e["gdal"], "· rasterio:", e["rasterio"], "·", e["runtime"], "· Pyodide" if e["pyodide"] else "")